In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.utils import negative_sampling
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, balanced_accuracy_score
from scipy.special import expit
import numpy as np
import os

def train(model, data, optimizer, device):
    """Train the model on the full graph."""
    model.train()
    optimizer.zero_grad()
    
    # Forward pass on the entire graph
    out = model(data.x_dict, data.edge_index_dict)
    drug_emb = out['chemical']
    disease_emb = out['disease']
    
    # Positive edges
    pos_edge_index = data['chemical', 'chem_disease', 'disease'].edge_index
    if pos_edge_index.size(1) == 0:
        raise ValueError("No chemical-disease edges found in the graph.")
    pos_score = predict_edges(drug_emb, disease_emb, pos_edge_index)
    
    # Negative edges
    neg_edge_index = negative_sampling(
        edge_index=pos_edge_index,
        num_nodes=(data['chemical'].num_nodes, data['disease'].num_nodes),
        num_neg_samples=pos_edge_index.size(1)
    )
    neg_score = predict_edges(drug_emb, disease_emb, neg_edge_index)
    
    # Binary cross-entropy loss
    scores = torch.cat([pos_score, neg_score])
    labels = torch.cat([torch.ones(pos_score.size(0)), torch.zeros(neg_score.size(0))]).to(device)
    loss = F.binary_cross_entropy_with_logits(scores, labels)
    
    loss.backward()
    optimizer.step()
    return loss.item()

def evaluate(model, data, device):
    """Evaluate the model on the full graph."""
    model.eval()
    with torch.no_grad():
        # Forward pass
        out = model(data.x_dict, data.edge_index_dict)
        drug_emb = out['chemical']
        disease_emb = out['disease']
        
        # Positive edges
        pos_edge_index = data['chemical', 'chem_disease', 'disease'].edge_index
        if pos_edge_index.size(1) == 0:
            raise ValueError("No chemical-disease edges found for evaluation.")
        pos_scores = predict_edges(drug_emb, disease_emb, pos_edge_index)
        pos_labels = torch.ones(pos_edge_index.size(1), device=device)
        
        # Negative edges
        neg_edge_index = negative_sampling(
            edge_index=pos_edge_index,
            num_nodes=(data['chemical'].num_nodes, data['disease'].num_nodes),
            num_neg_samples=pos_edge_index.size(1)
        )
        neg_scores = predict_edges(drug_emb, disease_emb, neg_edge_index)
        neg_labels = torch.zeros(neg_edge_index.size(1), device=device)
        
        # Combine scores and labels
        scores = torch.cat([pos_scores, neg_scores]).cpu().numpy()
        labels = torch.cat([pos_labels, neg_labels]).cpu().numpy()
        
        # Stable sigmoid for probabilities
        probs = expit(scores)
        predictions = (probs > 0.5).astype(int)
        
        # Compute metrics
        return {
            'roc_auc': roc_auc_score(labels, probs),
            'precision': precision_score(labels, predictions, zero_division=0),
            'recall': recall_score(labels, predictions, zero_division=0),
            'f1': f1_score(labels, predictions, zero_division=0),
            'balanced_accuracy': balanced_accuracy_score(labels, predictions)
        }

def predict_top_drugs(model, data, disease_id, chem_idx, disease_idx, top_k=5, device='cpu'):
    """Predict top-k drugs for a given disease ID."""
    model.eval()
    if disease_id not in disease_idx:
        raise ValueError(f"Disease ID {disease_id} not found in disease_idx.")
    
    disease_idx_val = disease_idx[disease_id]
    with torch.no_grad():
        # Forward pass on the full graph
        out = model(data.x_dict, data.edge_index_dict)
        drug_emb = out['chemical']
        disease_emb = out['disease']
        
        # Compute scores for all chemicals with the given disease
        disease_emb_single = disease_emb[disease_idx_val].unsqueeze(0)
        scores = (drug_emb * disease_emb_single).sum(dim=1).cpu().numpy()
        
        # Get top-k indices and scores
        top_k_indices = np.argsort(scores)[::-1][:top_k]
        top_k_scores = scores[top_k_indices]
        
        # Map indices to chemical IDs
        idx_to_chem = {i: c for c, i in chem_idx.items()}
        top_k_drugs = [(idx_to_chem[idx], score, expit(score)) for idx, score in zip(top_k_indices, top_k_scores)]
        
        return top_k_drugs

# Training and evaluation
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = HeteroGNN(hidden_channels=64, metadata=data.metadata()).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
data = data.to(device)

# Training loop
os.makedirs('checkpoints', exist_ok=True)
num_epochs = 50
try:
    for epoch in range(num_epochs):
        loss = train(model, data, optimizer, device)
        print(f'Epoch {epoch+1:03d}, Loss: {loss:.4f}')
        
        # Save checkpoint
        checkpoint = {
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': loss
        }
        torch.save(checkpoint, f'checkpoints/model_epoch_{epoch+1:03d}.pt')
    
    print("Training complete! Models saved in 'checkpoints' directory.")
    
    # Evaluate the model
    metrics = evaluate(model, data, device)
    print("\nEvaluation Metrics:")
    print(f"ROC-AUC: {metrics['roc_auc']:.4f}")
    print(f"Precision: {metrics['precision']:.4f}")
    print(f"Recall: {metrics['recall']:.4f}")
    print(f"F1 Score: {metrics['f1']:.4f}")
    print(f"Balanced Accuracy: {metrics['balanced_accuracy']:.4f}")
    
    # Predict top drugs for a given disease ID
    disease_id = diseases[0]  # Replace with your specific disease ID
    top_k = 5
    top_drugs = predict_top_drugs(model, data, disease_id, chem_idx, disease_idx, top_k=top_k, device=device)
    print(f"\nTop {top_k} drugs for disease {disease_id}:")
    for drug_id, score, prob in top_drugs:
        print(f"Drug: {drug_id}, Score: {score:.4f}, Probability: {prob:.4f}")

except torch.cuda.OutOfMemoryError:
    print("GPU out of memory, switching to CPU...")
    device = torch.device('cpu')
    model = model.to(device)
    data = data.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
    
    # Load the last checkpoint if available, else restart
    checkpoint_path = f'checkpoints/model_epoch_{num_epochs:03d}.pt'
    if os.path.exists(checkpoint_path):
        checkpoint = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        print(f"Loaded checkpoint from {checkpoint_path}, epoch {checkpoint['epoch']}, loss: {checkpoint['loss']:.4f}")
    
    # Evaluate and predict on CPU
    metrics = evaluate(model, data, device)
    print("\nEvaluation Metrics:")
    print(f"ROC-AUC: {metrics['roc_auc']:.4f}")
    print(f"Precision: {metrics['precision']:.4f}")
    print(f"Recall: {metrics['recall']:.4f}")
    print(f"F1 Score: {metrics['f1']:.4f}")
    print(f"Balanced Accuracy: {metrics['balanced_accuracy']:.4f}")
    
    top_drugs = predict_top_drugs(model, data, disease_id, chem_idx, disease_idx, top_k=top_k, device=device)
    print(f"\nTop {top_k} drugs for disease {disease_id}:")
    for drug_id, score, prob in top_drugs:
        print(f"Drug: {drug_id}, Score: {score:.4f}, Probability: {prob:.4f}")